# Latent Representations of MALDI-TOF Data with PCA

- Baseline models show that some antibiotics are not label-limited but feature-limited, with pseudo-labeling providing little or no benefit.

- PCA is used to learn a latent representation of the MALDI-TOF spectra that reduces noise and redundancy while preserving biological signal.

- PCA is applied in a CV-safe manner: fitted only on training folds and then applied to validation folds to avoid leakage.

- Models are trained using species_id + PCA(MALDI) and compared directly against the raw-feature baseline under identical CV and hyperparameters.

- PCA is retained only if it yields a robust OOF AUC improvement (≥ +0.002) on antibiotics with moderate baseline performance.

In [3]:
!pip install catboost

In [4]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool
import pandas as pd
import numpy as np

In [6]:
data_folder = '/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof'

In [7]:
train = pd.read_csv(f"{data_folder}/train.csv")
test  = pd.read_csv(f"{data_folder}/test.csv")
sub   = pd.read_csv(f"{data_folder}/sample_submission.csv")

print(train.shape)
print(test.shape)
print(sub.shape)
train.head(2)

(3360, 6010)
(1000, 6002)
(1000, 9)


,sample_id,species_id,maldi_feature_0,maldi_feature_1,maldi_feature_2,maldi_feature_3,maldi_feature_4,maldi_feature_5,maldi_feature_6,maldi_feature_7,...,maldi_feature_5998,maldi_feature_5999,Ampicillin,Levofloxacin,Ciprofloxacin,Imipenem,Amoxicillin_Clavulanic_acid,Ertapenem,Cefotaxime,Cefuroxime
0,SAMPLE_00000,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,SAMPLE_00001,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [8]:
# CV-safe PCA(50) experiment for 4 targets:
# Ampicillin, Ciprofloxacin, Levofloxacin, Amoxicillin_Clavulanic_acid
#
# Key rule: PCA is fit on TRAIN-fold only (no leakage), then applied to VAL-fold.
# Features used: species_id (categorical) + PCA(MALDI).
#
# Assumes you already have:
#   train = pd.read_csv(...)
# and CatBoost + sklearn installed.

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier

# ----------------------------
# Config
# ----------------------------
TARGETS_TO_TEST = [
    "Ampicillin",
    "Ciprofloxacin",
    "Levofloxacin",
    "Amoxicillin_Clavulanic_acid",
]

N_SPLITS = 5
SEED = 42
N_PCA = 50           # change to 20/100 to probe
USE_GPU = True      # keep False for reproducibility; GPU can be non-deterministic

# CatBoost factory (yours)
def make_cb(seed=42, use_gpu=False):
    params = dict(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=5000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=seed,
        verbose=False,
        od_type="Iter",
        od_wait=200,
    )
    if use_gpu:
        params.update(task_type="GPU")
    return CatBoostClassifier(**params)

# ----------------------------
# Columns
# ----------------------------
SPECIES_COL = "species_id"
maldi_cols = [c for c in train.columns if c.startswith("maldi_feature_")]
if len(maldi_cols) == 0:
    raise ValueError("No MALDI columns found with prefix 'maldi_feature_'")

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# ----------------------------
# Helpers
# ----------------------------
def cv_auc_catboost_raw(target: str):
    """Baseline: species_id + RAW MALDI (no PCA)."""
    labeled = train.dropna(subset=[target]).copy()
    X = labeled[[SPECIES_COL] + maldi_cols]
    y = labeled[target].astype(int)

    oof = np.zeros(len(labeled), dtype=float)
    fold_aucs = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = make_cb(seed=SEED + fold, use_gpu=USE_GPU)
        model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            cat_features=[SPECIES_COL],
            use_best_model=True,
            verbose=False,
        )
        p_va = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = p_va
        fold_auc = roc_auc_score(y_va, p_va)
        fold_aucs.append(fold_auc)

    return float(roc_auc_score(y, oof)), float(np.mean(fold_aucs)), float(np.std(fold_aucs))


def cv_auc_catboost_pca(target: str, n_pca: int = 50):
    """Experiment: species_id + PCA(MALDI) with PCA fit on train-fold only."""
    labeled = train.dropna(subset=[target]).copy()

    # Keep raw parts separate to avoid accidental leakage
    X_species = labeled[[SPECIES_COL]].reset_index(drop=True)
    X_maldi = labeled[maldi_cols].reset_index(drop=True)
    y = labeled[target].astype(int).reset_index(drop=True)

    oof = np.zeros(len(labeled), dtype=float)
    fold_aucs = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_species, y), start=1):
        # Split
        sp_tr, sp_va = X_species.iloc[tr_idx], X_species.iloc[va_idx]
        maldi_tr, maldi_va = X_maldi.iloc[tr_idx], X_maldi.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        # CV-safe PCA: fit only on train-fold MALDI
        pca = PCA(n_components=n_pca, random_state=SEED)
        Z_tr = pca.fit_transform(maldi_tr.values)
        Z_va = pca.transform(maldi_va.values)

        # Build fold featureframes: species_id + pca_*
        Z_tr_df = pd.DataFrame(Z_tr, columns=[f"pca_{i}" for i in range(n_pca)], index=sp_tr.index)
        Z_va_df = pd.DataFrame(Z_va, columns=[f"pca_{i}" for i in range(n_pca)], index=sp_va.index)

        X_tr = pd.concat([sp_tr, Z_tr_df], axis=1)
        X_va = pd.concat([sp_va, Z_va_df], axis=1)

        model = make_cb(seed=SEED + fold, use_gpu=USE_GPU)
        model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            cat_features=[SPECIES_COL],
            use_best_model=True,
            verbose=False,
        )
        p_va = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = p_va
        fold_auc = roc_auc_score(y_va, p_va)
        fold_aucs.append(fold_auc)

    return float(roc_auc_score(y, oof)), float(np.mean(fold_aucs)), float(np.std(fold_aucs))


# ----------------------------
# Run experiments
# ----------------------------
results = []

for t in TARGETS_TO_TEST:
    print(f"\n=== {t} ===")

    base_oof, base_mean, base_std = cv_auc_catboost_raw(t)
    print(f"Baseline (raw)  OOF AUC: {base_oof:.5f} | mean±std folds: {base_mean:.5f} ± {base_std:.5f}")

    pca_oof, pca_mean, pca_std = cv_auc_catboost_pca(t, n_pca=N_PCA)
    print(f"PCA({N_PCA})     OOF AUC: {pca_oof:.5f} | mean±std folds: {pca_mean:.5f} ± {pca_std:.5f}")

    delta = pca_oof - base_oof
    print(f"Delta (PCA - raw): {delta:+.5f}")

    results.append({
        "target": t,
        "raw_oof_auc": base_oof,
        f"pca{N_PCA}_oof_auc": pca_oof,
        "delta": delta,
        "raw_fold_std": base_std,
        f"pca{N_PCA}_fold_std": pca_std
    })

res_df = pd.DataFrame(results).sort_values("delta", ascending=False)
print("\n=== Summary (sorted by delta) ===")
print(res_df.to_string(index=False))



=== Ampicillin ===


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


Baseline (raw)  OOF AUC: 0.87043 | mean±std folds: 0.93271 ± 0.00635


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


PCA(50)     OOF AUC: 0.86312 | mean±std folds: 0.93226 ± 0.00545
Delta (PCA - raw): -0.00731

=== Ciprofloxacin ===


...

PCA(50)     OOF AUC: 0.81280 | mean±std folds: 0.81446 ± 0.00878
Delta (PCA - raw): -0.03754

=== Amoxicillin_Clavulanic_acid ===


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


Baseline (raw)  OOF AUC: 0.72536 | mean±std folds: 0.73486 ± 0.02773


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


PCA(50)     OOF AUC: 0.67054 | mean±std folds: 0.69829 ± 0.03032
Delta (PCA - raw): -0.05482

=== Summary (sorted by delta) ===
                     target  raw_oof_auc  pca50_oof_auc     delta  raw_fold_std  pca50_fold_std
                 Ampicillin     0.870432       0.863121 -0.007311      0.006347        0.005455
               Levofloxacin     0.850340       0.812797 -0.037544      0.007053        0.008777
Amoxicillin_Clavulanic_acid     0.725362       0.670543 -0.054819      0.027728        0.030319
              Ciprofloxacin     0.852864       0.794614 -0.058250      0.020218        0.029620
